# Step 8: statistics (point 3), leave-one-cohort-out (point 4), final test (point 4, OFF by default)
**Point 3, statistics** on the validation predictions of every model: AUC with DeLong 95% CI, AUPRC, Brier,
ECE, sensitivity / PPV / NPV at 90% specificity, the paired DeLong test vs the clinical baseline B1,
calibration (reliability) plots, and AUC within subtypes and cohorts. It reads the repo's committed results plus
the attached step-7 (and step-6) outputs.

**Point 4a, leave-one-cohort-out (LOCO):** for each cohort, our ViT-only and fusion models (the best setting
from step 7, read from its `best_setting.json`) are trained on the other two cohorts (early stopping on their
val), then score the held-out cohort's train + val patients, which they never saw. B1 is done the same way.

**Point 4b, the final test evaluation:** **once**, on the frozen final models, with thresholds from val.
`FINAL_TEST = False` by default. Agree with the team first (plan D-006: the test set is used once; every use
is logged). It needs the step-7 output attached (its checkpoints).

Kaggle setup: GPU T4 x2 (for LOCO), Internet ON. **+ Add Input**: the caches (`mnj-cache-ispy1/-ispy2/-duke`),
the code (`mnj-code-vaib` new version, or the `GITHUB_TOKEN` secret), and **your step-7 notebook's output**
(+ Add Input -> Your Work -> Notebooks).

In [ ]:
# ==== PARAMETERS (edit these) ====
REPO = "AkulaReddy/MRI_chemo_response"
GIT_TAG = "vaib"           # branch (or tag) to clone
PERSON = ""               # optional: "A", "B" or "C" -> results/<PERSON>/ (empty -> results/vaib/)
STEP = "step8"
CACHE_VERSION = None
RUN_STATS = True
REFERENCE = "B1 clinical + size (LR)"
MODELS = {   # name: run-folder name pattern (glob) under results/ of the repo or the attached outputs; seeds averaged
    "B1 clinical + size (LR)":              "b1_lr_s0_*",
    "B0 clinical (LR)":                     "b0_lr_s0_*",
    "B3 ViT, old pipeline":                 "b3_vit_s0_1009-1616_*",
    "B3 ViT, cache v2 (Akula)":             "b3_vit_s0_1019-0302_*",
    "B3 ViT, cache v2, 2 blocks (Akula)":   "b3_vit_s0_1019-0500_*",
    "F2 fusion, old pipeline":              "f2_attn_s0_1009-1605_*",
    "ViT-only ours, step 5 (mae+kinetic)":  "vo_base_mae_kinetic_tumor_s*",
    "ViT-only ours, best (step 7)":         "step7_vitonly_best_ensemble",
    "ViT-only ours, best regularised":      "step7_sweep_best",
    "Fusion ours (step 7)":                 "step7_fusion_ensemble",
}
RUN_LOCO = True
LOCO_MODELS = {"ViT-only ours": "vo_base", "Fusion ours": "vo_fusion"}
LOCO_SEED = 0
LOCO_SETS = None             # None = the best setting from the attached step-7 output; or a list of overrides
FINAL_TEST = False           # True ONCE, with the team's agreement
FINAL_CONFIRM = ""           # must be "I use the test set once" when FINAL_TEST = True
FINAL_MODELS = {             # name: "step:key" of runs.json written by step 7 (checkpoints from the attached output)
    "Fusion ours": "step7:fusion",
    "ViT-only ours, best": "step7:vitonly_best",
}
TIME_BUDGET_H = 11.0
DEBUG = False
PUSH_RESULTS = False
NOTEBOOK_NAME = "step8"

In [ ]:
# ==== SETUP (do not edit): get the repo code, install requirements ====
import base64, json, os, shutil, subprocess, sys
if PERSON not in ("A", "B", "C"):           # only names the results folder
    PERSON = "vaib"
    print('PERSON not set: results go to results/vaib/ (set PERSON = "A", "B" or "C" to change)')
_tok = None
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as e:
    print("no GITHUB_TOKEN secret attached to this notebook (fine if the mnj-code-vaib Dataset is attached):",
          type(e).__name__)

def git(*args, cwd="/tmp/repo"):
    """git with the token as a one-off header (never stored in .git/config or the output)."""
    if not _tok:
        raise RuntimeError("no GITHUB_TOKEN secret: git push/pull is not available")
    hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
    r = subprocess.run(["git", "-c", f"http.extraHeader={hdr}", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_tok, "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

shutil.rmtree("/tmp/repo", ignore_errors=True)
if _tok:                                    # 1) clone the branch from GitHub
    git("clone", "--quiet", "--depth", "1", "--branch", GIT_TAG, f"https://github.com/{REPO}.git", "/tmp/repo", cwd="/tmp")
    print("code: GitHub", REPO, GIT_TAG, git("log", "-1", "--format=%h %s"))
else:                                       # 2) or a code Dataset (zip of the repo) attached as input
    import glob, zipfile
    hits = [h for d in range(1, 8) for h in glob.glob("/kaggle/input/" + "*/" * d + "src/vitonly/__init__.py")]
    zips = [z for d in range(1, 6) for z in glob.glob("/kaggle/input/" + "*/" * d + "mnj-code-vaib*.zip")]
    if not hits and zips:                   # Kaggle kept the uploaded zip as a file: unpack it
        zipfile.ZipFile(zips[0]).extractall("/tmp/code_zip")
        hits = glob.glob("/tmp/code_zip/**/src/vitonly/__init__.py", recursive=True)
    if not hits:
        print("attached inputs:")
        for root, dirs, files in os.walk("/kaggle/input"):
            if root.count("/") <= 5:
                print("  ", root, dirs[:6], files[:4])
            else:
                dirs[:] = []
        raise SystemExit(
            "\nThe notebook can't find the project code. Do ONE of these, then Run All again:\n"
            "  (a) Add-ons -> Secrets -> add GITHUB_TOKEN (a GitHub token with access to the repo) and switch it ON\n"
            "      for this notebook, or\n"
            "  (b) + Add Input -> Datasets -> Your Datasets -> mnj-code-vaib (made by uploading mnj-code-vaib.zip).")
    code_root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    shutil.copytree(code_root, "/tmp/repo")
    print("code: attached Dataset", code_root)
os.chdir("/tmp/repo")
sys.path.insert(0, "/tmp/repo")         # so `from src...` works in this kernel
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# ==== HELPERS (do not edit): training runs, prediction files, comparisons ====
import glob, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import evaluation as ev, stats
from src.train import find_caches
from src.data.metadata import patient_table
COH = ["ispy1", "ispy2", "duke"]
OUT = f"/kaggle/working/results/{PERSON}/{STEP}"
os.makedirs(OUT, exist_ok=True)
T0 = time.time()
RUNS_JSON = {}
pd.set_option("display.width", 250, "display.max_columns", 40)

def budget_left():
    return TIME_BUDGET_H - (time.time() - T0) / 3600

def check_caches(version):
    caches = find_caches(["/kaggle/input", "/kaggle/working"])
    info = {c: json.load(open(f"{d}/cache_meta.json")).get("builder_version", 1) for c, d in caches.items()}
    print("caches:", {c: f"builder v{v}: {caches[c]}" for c, v in info.items()})
    if version is not None:
        bad = {c: v for c, v in info.items() if v != version}
        assert not bad and len(info) == 3, (f"need cache builder v{version} for all 3 cohorts (attach mnj-cache-v2); "
                                            f"found {info}")
    return caches

def _copy(d, label):
    os.makedirs(f"{OUT}/{label}", exist_ok=True)
    for f in ("val_preds.csv", "metrics.json", "log.csv", "config_resolved.yaml", "export_preds.csv",
              "train_preds.csv", "encoder.json"):
        if os.path.exists(f"{d}/{f}"):
            shutil.copy(f"{d}/{f}", f"{OUT}/{label}/{f}")

def train(config, seed, sets=(), label=None, cohorts=None):
    """One src.train run (B3 / F2 ...) -> (run_dir, metrics). Its small files are copied to OUT/<label>/."""
    cmd = [sys.executable, "-m", "src.train", "--config", config, "--seed", str(seed), "--out", "/kaggle/working/runs"]
    if CACHE_VERSION is not None: cmd += ["--cache-version", str(CACHE_VERSION)]
    if cohorts: cmd += ["--cohorts", *cohorts]
    if DEBUG: cmd.append("--debug")
    if sets: cmd += ["--set", *sets]
    print("\n" + " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    d = max(glob.glob("/kaggle/working/runs/*/"), key=os.path.getmtime).rstrip("/")
    if label: _copy(d, label)
    return d, json.load(open(f"{d}/metrics.json"))

def finetune(config, seed, sets=(), label=None, cohorts=None):
    """One src.vitonly.finetune run (our pipeline: ViT-only vo_base / fusion vo_fusion) -> run_dir."""
    cmd = [sys.executable, "-m", "src.vitonly.finetune", "--config", config, "--seed", str(seed),
           "--cache-root", "/kaggle/input", "/kaggle/working", "--out", "/kaggle/working/runs"]
    if cohorts: cmd += ["--cohorts", *cohorts]
    if DEBUG: cmd.append("--debug")
    if sets: cmd += ["--set", *sets]
    print("\n" + " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    d = max(glob.glob("/kaggle/working/runs/*/"), key=os.path.getmtime).rstrip("/")
    if label: _copy(d, label)
    return d

def note_runs(key, dirs):
    """label -> run folder names, in OUT/runs.json: step 8 uses it to find the checkpoints."""
    RUNS_JSON[key] = [os.path.basename(d) for d in dirs]
    json.dump(RUNS_JSON, open(f"{OUT}/runs.json", "w"), indent=1)

def preds_of(dirs):
    fr = [ev.load_preds(f"{d}/val_preds.csv") for d in dirs]
    return fr[0] if len(fr) == 1 else ev.ensemble(fr)

def save_preds(label, df):
    os.makedirs(f"{OUT}/{label}", exist_ok=True)
    df.to_csv(f"{OUT}/{label}/val_preds.csv", index=False)
    return df

def repo_preds(pattern):
    hits = sorted(glob.glob(f"results/*/{pattern}/val_preds.csv"))
    return preds_of([os.path.dirname(h) for h in hits]) if hits else None

def train_auc(d):
    """AUC on the training patients from export_preds.csv (B3 exports after training): the overfitting gap."""
    f = f"{d}/export_preds.csv"
    if not os.path.exists(f): return np.nan
    e = pd.read_csv(f); e = e[e.split == "train"]
    return stats.auc(e.label, e.prob)

def vo_train_auc(d):
    """Training-set AUC of a src.vitonly run (metrics.json): the overfitting gap."""
    return json.load(open(f"{d}/metrics.json"))["val"].get("train_auc", np.nan)

def compare(preds, ref):
    """One row per model on the patients all models share: AUC (DeLong CI), AUPRC, Brier, ECE, the 90%-spec
    operating point (threshold chosen on these same val patients: optimistic), and the paired DeLong test
    and paired bootstrap of the AUC difference vs `ref`."""
    preds = ev.align({k: v for k, v in preds.items() if v is not None})
    rows = []
    for k, d in preds.items():
        r = {"model": k, **ev.summary_row(d.label, d.prob)}
        if k != ref and ref in preds:
            y = preds[ref].label
            dl = ev.delong(y, preds[ref].prob, d.prob)
            pb = stats.paired_bootstrap_auc_diff(y, preds[ref].prob, d.prob)
            r.update(diff_vs_ref=dl["diff"], delong_ci_lo=dl["ci"][0], delong_ci_hi=dl["ci"][1], delong_p=dl["p"],
                     boot_p=pb["p"])
        rows.append(r)
    cols = ["model", "n", "pcr", "auc", "ci_lo", "ci_hi", "auprc", "brier", "ece", "sens@90spec", "ppv@90spec",
            "npv@90spec", "diff_vs_ref", "delong_ci_lo", "delong_ci_hi", "delong_p", "boot_p"]
    t = pd.DataFrame(rows)
    return t[[c for c in cols if c in t]]

def subtype_wide(preds, by="subtype"):
    """AUC (95% CI) per subtype (or cohort) for every model, all on the same patients; n / pCR+ per group."""
    preds = ev.align({k: v for k, v in preds.items() if v is not None})
    tabs = {k: ev.subgroup_table(d, by).set_index(by) for k, d in preds.items()}
    first = next(iter(tabs.values()))
    out = pd.DataFrame({"n": first.n.astype(int), "pCR+": first.pcr.astype(int)})
    for k, t in tabs.items():
        out[k] = [f"{a:.3f} ({lo:.2f}-{hi:.2f})" if a == a else "n/a (one class)"
                  for a, lo, hi in zip(t.auc, t.ci_lo, t.ci_hi)]
    return out

B1_VAL = repo_preds("b1_lr_s0_*")                       # clinical + tumour size, logistic regression (val 0.653)
B0_VAL = repo_preds("b0_lr_s0_*")                       # clinical only (val 0.639)
AKULA_B3_V2 = repo_preds("b3_vit_s0_1019-0302_*")       # Akula's B3 on cache v2 (val 0.559)
B3_V1 = repo_preds("b3_vit_s0_1009-1616_*")             # B3 on the old cache (val 0.530)
F2_V1 = repo_preds("f2_attn_s0_1009-1605_*")            # fusion on the old cache (val 0.690)
print("reference predictions from the repo:", {k: v is not None for k, v in
      dict(B1=B1_VAL, B0=B0_VAL, B3_v2_Akula=AKULA_B3_V2, B3_v1=B3_V1, F2_v1=F2_V1).items()})

## Point 3: statistics on validation

In [ ]:
ROOTS = ["results", "/kaggle/input"]
def collect(pattern):
    seen, hits = set(), []
    for r in ROOTS:
        for h in sorted(glob.glob(f"{r}/**/{pattern}/val_preds.csv", recursive=True)):
            name = os.path.basename(os.path.dirname(h))
            if name not in seen:            # the same run can sit in the repo and in an attached output
                seen.add(name); hits.append(h)
    return hits

if RUN_STATS:
    preds = {}
    for name, pat in MODELS.items():
        hits = collect(pat)
        if not hits:
            print(f"missing (not run / not attached): {name} [{pat}]"); continue
        preds[name] = preds_of([os.path.dirname(h) for h in hits])
        print(f"{name}: {len(hits)} run(s)")
    tab = compare(preds, REFERENCE)
    tab.to_csv(f"{OUT}/step8_val_statistics.csv", index=False)
    display(tab.round(3))
    print("sens/ppv/npv@90spec: threshold chosen on these val patients (optimistic); the final test uses val thresholds.")
    al = ev.align(preds)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ev.plot_reliability(ax[0], {k: (v.label, v.prob) for k, v in al.items() if "B1" in k or "F2" in k or "B0" in k})
    ev.plot_reliability(ax[1], {k: (v.label, v.prob) for k, v in al.items() if "ViT" in k or "B3" in k},
                        "Calibration, image-only models (val)")
    plt.tight_layout(); plt.savefig(f"{OUT}/step8_calibration.png", dpi=110); plt.show()
    st = subtype_wide(preds); st.to_csv(f"{OUT}/step8_val_by_subtype.csv"); print("AUC within subtypes:"); display(st)
    sc = subtype_wide(preds, "cohort"); sc.to_csv(f"{OUT}/step8_val_by_cohort.csv"); print("AUC per cohort:"); display(sc)

## Point 4a: leave-one-cohort-out (our ViT-only and fusion models, B1)

In [ ]:
if RUN_LOCO:
    import re
    caches = check_caches(CACHE_VERSION)
    from src.predict_run import score_run
    bs = {}
    if LOCO_SETS is None:
        found = sorted(glob.glob("/kaggle/input/**/best_setting.json", recursive=True))
        bs = json.load(open(found[0])) if found else {}
        sets = bs.get("overrides", ["model.backbone=dinov2_b", "optim.lr_backbone=1.0e-5", "data.norm=sub",
                                    "data.crop=tumor", "model.pool=gap"])
        print("LOCO setting:", bs.get("name", "default: dinov2_b + sub + tumour crop (attach the step-7 output)"), sets)
    else:
        sets = list(LOCO_SETS)
    fsets = bs.get("fusion_overrides", sets)
    rows = []
    for held in COH:
        others = [c for c in COH if c != held]
        tr = patient_table("train", others, cache_dirs=caches)               # same patients as the image models
        sc_t = patient_table(["train", "val"], [held], cache_dirs=caches)
        b1 = ev.clinical_preds(tr, sc_t, "b1_lr")
        r = ev.delong_ci(b1.label, b1.prob)
        rows.append({"model": "B1 clinical + size (LR)", "held_out": held, "n": len(b1), "pcr": int(b1.label.sum()),
                     "auc": r["auc"], "ci_lo": r["ci"][0], "ci_hi": r["ci"][1]})
        for name, cfg in LOCO_MODELS.items():
            if budget_left() < 0.9:
                print(f"{name} / {held} skipped: time budget"); continue
            label = "step8_loco_" + re.sub(r"[^a-z0-9]+", "_", name.lower()) + f"_{held}"
            d = finetune(cfg, LOCO_SEED, fsets if cfg == "vo_fusion" else sets, label, cohorts=others)
            m = json.load(open(f"{d}/metrics.json"))
            p = score_run(d, [held], ["train", "val"])
            p.to_csv(f"{OUT}/{label}/heldout_preds.csv", index=False)
            r = ev.delong_ci(p.label, p.prob)
            rows.append({"model": name, "held_out": held, "n": len(p), "pcr": int(p.label.sum()), "auc": r["auc"],
                         "ci_lo": r["ci"][0], "ci_hi": r["ci"][1], "val_auc_training_cohorts": m["val"]["auc"]})
            print(f"{name} trained on {others} -> {held}: AUC {r['auc']:.3f} ({r['ci'][0]:.2f}-{r['ci'][1]:.2f})")
    loco = pd.DataFrame(rows)
    loco.to_csv(f"{OUT}/step8_loco.csv", index=False)
    display(loco.round(3))
    display(loco.pivot(index="model", columns="held_out", values="auc").round(3))

## Point 4b: final test evaluation (once)

In [ ]:
if FINAL_TEST:
    assert FINAL_CONFIRM == "I use the test set once", 'set FINAL_CONFIRM = "I use the test set once"'
    from src.predict_run import score_run
    rj = {}
    for f in glob.glob("/kaggle/input/**/runs.json", recursive=True) + glob.glob("/kaggle/working/results/**/runs.json", recursive=True):
        step = os.path.basename(os.path.dirname(f))
        rj.setdefault(step, {}).update(json.load(open(f)))
    all_runs = {d.name: d for d in ev.find_run_dirs(["/kaggle/input", "/kaggle/working"])}
    train_t, val_t = patient_table("train", COH), patient_table("val", COH)
    test_t = patient_table("test", COH, allow_test=True)
    final, log = {}, []
    b1_val = ev.clinical_preds(train_t, val_t, "b1_lr")
    final["B1 clinical + size (LR)"] = (ev.clinical_preds(train_t, test_t, "b1_lr"),
                                        ev.threshold_at_spec(b1_val.label, b1_val.prob))
    for name, key in FINAL_MODELS.items():
        step, k = key.split(":")
        ids = rj.get(step, {}).get(k, [])
        dirs = [all_runs[i] for i in ids if i in all_runs]
        if not dirs:
            print(f"SKIPPED {name}: no checkpoints for {key} (attach the {step} notebook output; "
                  "a model reused from the repo has predictions but no checkpoint)"); continue
        tests = [score_run(d, COH, ["test"], final=True) for d in dirs]
        log += [{"time": time.strftime("%Y-%m-%d %H:%M:%S"), "model": name, "run": d.name, "n": len(t)} for d, t in zip(dirs, tests)]
        vals = preds_of(dirs)
        final[name] = (ev.ensemble(tests) if len(tests) > 1 else tests[0], ev.threshold_at_spec(vals.label, vals.prob))
    fdir = f"/kaggle/working/results/{PERSON}/final_test"; os.makedirs(fdir, exist_ok=True)
    pd.DataFrame(log).to_csv(f"{fdir}/final_eval_log.csv", mode="a", index=False,
                             header=not os.path.exists(f"{fdir}/final_eval_log.csv"))
    aligned = ev.align({k: v[0] for k, v in final.items()})       # the test patients every model scored
    ref = aligned["B1 clinical + size (LR)"]
    rows = []
    for name, (_, t) in final.items():
        p = aligned[name]
        r = {"model": name, **ev.summary_row(p.label, p.prob, t=t)}
        if name != "B1 clinical + size (LR)":
            dl = ev.delong(ref.label, ref.prob, p.prob)
            r.update(diff_vs_B1=dl["diff"], delong_ci_lo=dl["ci"][0], delong_ci_hi=dl["ci"][1], delong_p=dl["p"])
        rows.append(r)
        p.to_csv(f"{fdir}/{name.replace(' ', '_').replace(',', '')}_test_preds.csv", index=False)
    ft = pd.DataFrame(rows); ft.to_csv(f"{fdir}/final_test_statistics.csv", index=False)
    print(f"TEST SET: {len(ref)} patients, {int(ref.label.sum())} pCR. Thresholds from val (90% spec)."); display(ft.round(3))
    fp = aligned
    st = subtype_wide(fp); st.to_csv(f"{fdir}/final_test_by_subtype.csv"); display(st)
    sc = subtype_wide(fp, "cohort"); sc.to_csv(f"{fdir}/final_test_by_cohort.csv"); display(sc)
    fig, ax = plt.subplots(figsize=(6.5, 5)); ev.plot_reliability(ax, {k: (v.label, v.prob) for k, v in fp.items()}, "Calibration (test)")
    plt.tight_layout(); plt.savefig(f"{fdir}/final_test_calibration.png", dpi=110); plt.show()
else:
    print("final test not run (FINAL_TEST = False)")

In [ ]:
# ==== PUSH the small result files to the vaib branch (results/<PERSON>/...) ====
if PUSH_RESULTS and not _tok:
    print("PUSH_RESULTS needs the GITHUB_TOKEN secret; download results/ from the Output tab instead")
elif PUSH_RESULTS:
    git("config", "user.name", f"Person {PERSON} (Kaggle)")
    git("config", "user.email", "kaggle@users.noreply.github.com")
    for attempt in range(3):
        git("fetch", "--quiet", "--depth", "1", "origin", f"+refs/heads/{GIT_TAG}:refs/remotes/origin/{GIT_TAG}")
        git("checkout", "--quiet", "-f", "-B", GIT_TAG, f"origin/{GIT_TAG}")
        shutil.copytree(f"/kaggle/working/results/{PERSON}", f"results/{PERSON}", dirs_exist_ok=True)
        git("add", f"results/{PERSON}")
        print(git("commit", "-m", f"results: {PERSON} {NOTEBOOK_NAME} ({GIT_TAG})"))
        try:
            git("push", "--quiet", "origin", GIT_TAG); print("pushed to", GIT_TAG); break
        except RuntimeError as e:
            print(f"push attempt {attempt + 1} failed:", e)
    else:
        print("NOT pushed: download results/ from the Output tab")